<a href="https://colab.research.google.com/github/nehansa2003/NLP_project/blob/main/NLP_reviews_scrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Scrape the Game name and profile links

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://opencritic.com"

all_games = []

for page in range(1, 101):
    url = f"https://opencritic.com/browse/all?page={page}"

    print(f"Scraping page {page}...")

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed page {page}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    game_rows = soup.find_all("div", class_="game-row")

    for row in game_rows:

        game_name_div = row.find("div", class_="game-name")

        if game_name_div:
            a_tag = game_name_div.find("a")

            if a_tag:
                game_name = a_tag.text.strip()

                game_link = BASE_URL + a_tag["href"]

                all_games.append({
                    "game_name": game_name,
                    "game_link": game_link
                })

    time.sleep(1)

df = pd.DataFrame(all_games)

df.to_csv("opencritic_games.csv", index=False)

print(f"Total games found: {len(df)}")
print(df.head())

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...
Scraping 

In [ ]:
from google.colab import files

files.download("opencritic_games.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.shape

(2000, 2)

Testing Scrape for one URL

In [ ]:
BASE_URL = "https://opencritic.com/game/4504/super-mario-odyssey/reviews"

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_reviews = []
page = 1

while True:

    if page == 1:
        url = BASE_URL
    else:
        url = f"{BASE_URL}?page={page}"

    print(f"Scraping page {page}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Failed:", url)
        break

    soup = BeautifulSoup(response.text, "html.parser")

    review_rows = soup.find_all("div", class_="row review-row py-2")

    # stop when empty page reached
    if len(review_rows) == 0:
        print("No more reviews found.")
        break

    for review in review_rows:

        # Author
        author_tag = review.select_one("app-author-list a")
        author = author_tag.get_text(strip=True) if author_tag else None

        # Outlet
        outlet_tag = review.select_one("span.outlet-name a")
        outlet = outlet_tag.get_text(strip=True) if outlet_tag else None

        # Date
        date_tag = review.select_one("div.date-block")
        date = date_tag.get_text(strip=True) if date_tag else None

        # Review text
        review_text = None
        review_link = None

        paragraphs = review.find_all("p")

        if len(paragraphs) > 0:
            review_text = paragraphs[0].get_text(" ", strip=True)

        read_more = review.find("a", string=lambda x: x and "Read full review" in x)

        if read_more:
            review_link = read_more.get("href")

        # Score
        score_div = review.select_one(".score-display")

        score = None

        if score_div:

            score_span = score_div.find("span", class_="score-number-bold")

            if score_span:
                score = score_span.get_text(strip=True)

            else:
                stars = len(score_div.select("i.fa-star"))

                if stars > 0:
                    score = f"{stars} stars"

        all_reviews.append({
            "author": author,
            "outlet": outlet,
            "score": score,
            "date": date,
            "review_text": review_text,
            "review_url": review_link
        })

    page += 1
    time.sleep(1)

# Save results
df = pd.DataFrame(all_reviews)

print("\nTotal Reviews:", len(df))
print(df.head())

df.to_csv("super_mario_odyssey_reviews.csv", index=False, encoding="utf-8-sig")

print("Saved to super_mario_odyssey_reviews.csv")

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
No more reviews found.

Total Reviews: 158
           author         outlet       score          date  \
0    Riley Little      Game Rant     5 stars  Oct 26, 2017   
1  Ryan McCaffrey            IGN   10 / 10.0  Oct 26, 2017   
2       Oli Welsh      Eurogamer   Essential  Oct 26, 2017   
3   Sam Loveridge    GamesRadar+     5 stars  Oct 26, 2017   
4   Andrew Reiner  Game Informer  9.8 / 10.0  Oct 26, 2017   

                                         review_text  \
0  Super Mario Odyssey is a spectacular return to...   
1  Mario's games have been around for almost as l...   
2  One of the most daring and influential game de...   
3  Super Mario Odyssey successfully brings the se...   
4  With hundreds of moons to collect and a dizzyi...   

                                          review_url  
0   https://gamerant.

Scraping the Whole Reviews in URLs

In [ ]:
import uuid

# Load game list
games_df = pd.read_csv("/content/drive/MyDrive/Outliers.ipynb/opencritic_games.csv")

# Add game IDs
games_df["game_id"] = range(1, len(games_df) + 1)

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_reviews = []

total_games = len(games_df)

for _, game in games_df.iterrows():

    game_id = game["game_id"]
    game_name = game["game_name"]
    game_url = game["game_link"]

    print(f"\n[{game_id}/{total_games}] Scraping: {game_name}")

    reviews_base_url = game_url.rstrip("/") + "/reviews"

    page = 1

    while True:

        if page == 1:
            url = reviews_base_url
        else:
            url = f"{reviews_base_url}?page={page}"

        try:

            response = requests.get(
                url,
                headers=headers,
                timeout=20
            )

            if response.status_code != 200:
                print(f"Failed page {page}")
                break

            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            review_rows = soup.find_all(
                "div",
                class_="row review-row py-2"
            )

            # Stop when page has no reviews
            if len(review_rows) == 0:
                print(f"Finished {game_name}")
                break

            print(
                f"Page {page}: {len(review_rows)} reviews"
            )

            for review in review_rows:

                author_tag = review.select_one(
                    "app-author-list a"
                )

                author = (
                    author_tag.get_text(strip=True)
                    if author_tag
                    else None
                )


                outlet_tag = review.select_one(
                    "span.outlet-name a"
                )

                outlet = (
                    outlet_tag.get_text(strip=True)
                    if outlet_tag
                    else None
                )

                date_tag = review.select_one(
                    "div.date-block"
                )

                date = (
                    date_tag.get_text(strip=True)
                    if date_tag
                    else None
                )


                review_text = None

                paragraphs = review.find_all("p")

                if len(paragraphs) > 0:
                    review_text = paragraphs[0].get_text(
                        " ",
                        strip=True
                    )


                review_url = None

                read_more = review.find(
                    "a",
                    string=lambda x:
                    x and "Read full review" in x
                )

                if read_more:
                    review_url = read_more.get("href")


                score = None

                score_div = review.select_one(
                    ".score-display"
                )

                if score_div:

                    score_span = score_div.find(
                        "span",
                        class_="score-number-bold"
                    )

                    if score_span:
                        score = score_span.get_text(
                            strip=True
                        )

                    else:
                        stars = len(
                            score_div.select(
                                "i.fa-star"
                            )
                        )

                        if stars > 0:
                            score = f"{stars} stars"


                review_id = str(uuid.uuid4())


                all_reviews.append({

                    "game_id": game_id,
                    "game_name": game_name,
                    "game_url": game_url,

                    "review_id": review_id,

                    "author": author,
                    "outlet": outlet,

                    "score": score,
                    "date": date,

                    "review_text": review_text,
                    "review_url": review_url
                })

            page += 1

            time.sleep(1)

        except Exception as e:

            print(
                f"Error on {game_name} page {page}:",
                e
            )

            break

    # Backup every 50 games
    if game_id % 50 == 0:

        backup_df = pd.DataFrame(all_reviews)

        backup_df.to_csv(
            "backup_reviews.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"Backup saved after {game_id} games"
        )


# Save Final Dataset

reviews_df = pd.DataFrame(all_reviews)

reviews_df.to_csv(
    "all_game_reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n================================")
print("SCRAPING COMPLETE")
print("================================")
print(f"Games Scraped : {total_games}")
print(f"Reviews Found : {len(reviews_df)}")
print("Saved as all_game_reviews.csv")

# Download automatically in Colab
from google.colab import files

files.download("all_game_reviews.csv")

Streaming output truncated to the last 5000 lines.
Page 4: 20 reviews
Page 5: 8 reviews
Finished Child of Light

[1109/2000] Scraping: Dragon Quest VII: Fragments of the Forgotten Past
Page 1: 20 reviews
Page 2: 20 reviews
Page 3: 4 reviews
Finished Dragon Quest VII: Fragments of the Forgotten Past

[1110/2000] Scraping: Assault Android Cactus
Page 1: 20 reviews
Page 2: 20 reviews
Page 3: 7 reviews
Finished Assault Android Cactus

[1111/2000] Scraping: Final Fantasy X / X-2 HD Remaster
Page 1: 20 reviews
Page 2: 20 reviews
Page 3: 20 reviews
Page 4: 3 reviews
Finished Final Fantasy X / X-2 HD Remaster

[1112/2000] Scraping: Viewfinder
Page 1: 20 reviews
Page 2: 20 reviews
Page 3: 20 reviews
Page 4: 20 reviews
Page 5: 15 reviews
Finished Viewfinder

[1113/2000] Scraping: In Stars and Time
Page 1: 20 reviews
Page 2: 1 reviews
Finished In Stars and Time

[1114/2000] Scraping: Death Stranding
Page 1: 20 reviews
Page 2: 20 reviews
Page 3: 20 reviews
Page 4: 20 reviews
Page 5: 20 reviews
Pag

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>